# H&M Personalized Fashion Recommendations

Multi-strategy retrieval → interaction features → LightGBM binary classifier.
The implementation lives in `src/hm_reco/`; this notebook walks through it step by step.
The same runs are available from the command line: `python train.py cv` / `python train.py submit --rounds N`.

In [1]:
import sys
sys.path.insert(0, "src")

import pandas as pd
from hm_reco import data, retrieval, pipeline, evaluation, submission

ds = data.load_dataset("data")   # first run parses the CSVs and caches parquet files
print(f"{len(ds.transactions):,} transactions, {len(ds.customers):,} customers, {ds.n_weeks} weeks")

31,788,324 transactions, 1,371,980 customers, 105 weeks


## 1. Retrieval

For a target week, every strategy only sees the weeks before it. Here: how many of the last week's
purchases each source recovers.

In [2]:
VAL_WEEK = ds.n_weeks - 1
week = pipeline.prepare_week(ds, VAL_WEEK)
labels = pipeline._labels(ds, VAL_WEEK)
buyers = ds.customers[ds.customers["customer_id"].isin(labels["customer_id"].unique())]

cand = retrieval.generate(week.ctx, buyers, ds.articles)
cand = cand.merge(labels, on=["customer_id", "article_id"], how="left")
print(f"{len(cand) / len(buyers):.0f} candidates per customer, "
      f"{int(cand['label'].sum()):,} of {len(labels):,} purchases retrieved")

sources = {"repurchase": "rep_days_ago", "itemcf": "cf_score", "siblings": "sib_sales",
           "popular": "pop_rank", "age popular": "agepop_rank"}
pd.DataFrame({name: {"candidates": int(cand[col].notna().sum()),
                     "hits": int(cand.loc[cand[col].notna(), "label"].sum())}
              for name, col in sources.items()}).T

104 candidates per customer, 29,114 of 213,728 purchases retrieved


,candidates,hits
repurchase,1456032,7441
itemcf,980771,4565
siblings,339937,6840
popular,3449200,15353
age popular,3449200,15546


## 2. Validation

Train on the 6 weeks before the last week (negatives downsampled), validate on the last week
(2020-09-16 .. 09-22), which mirrors the test week.

In [3]:
cfg = pipeline.Config(n_train_weeks=6, neg_per_week=600_000)
model, score = pipeline.run_cv(ds, cfg)

[17:40:46] building training set: target weeks 98..103


[17:41:42] week 98: 74,833 buyers, hits 27,105/259,512 (10.4% recall), train rows 627,105


[17:42:32] week 99: 71,094 buyers, hits 25,966/237,160 (10.9% recall), train rows 625,966


[17:43:18] week 100: 72,035 buyers, hits 28,295/230,825 (12.3% recall), train rows 628,295


[17:44:15] week 101: 80,253 buyers, hits 32,019/255,172 (12.5% recall), train rows 632,019


...

[17:50:31] scoring every validation-week buyer


[17:51:46]   scored 25,000/68,984 customers


[17:52:59]   scored 50,000/68,984 customers


[17:54:09]   scored 68,984/68,984 customers


[17:54:20] validation MAP@12: 0.03626  (popular-items baseline: 0.00959)


In [4]:
importance = pd.Series(model.feature_importance("gain"), index=model.feature_name())
importance.sort_values(ascending=False).head(25)

upc_tw_cnt           458783.413218
i_cnt_1w             305153.899157
department_no        274808.085911
ui_last_days         257576.151389
i_cnt_4w             110372.660669
upc_last_days         93886.105803
colour_group_code     79682.639271
product_type_no       75827.211663
u_channel_mean        61270.157184
udp_tw_cnt            59681.825719
n_sources             53045.258262
usc_share             51215.559947
udp_share             50959.565561
cf_score              50631.291443
i_channel_mean        49558.926437
u_last_days           45335.649092
i_trend               41948.564885
udp_last_days         38724.738152
ui_tw_cnt             35606.654385
u_active_days         33678.353374
agepop_rank           30506.015844
upc_cnt_4w            29636.329967
u_price_mean          28347.486503
sib_sales             27412.368475
upc_share             27070.925285
dtype: float64

## 3. Submission

Retrain with every target week shifted one week later, then score all 1.37M customers (about an hour on a laptop).
If `outputs/submission.csv` already exists (e.g. from `python train.py submit --rounds 434`), it's loaded instead.

In [5]:
import os

SUB_PATH = "outputs/submission.csv"
if not os.path.exists(SUB_PATH):
    preds = pipeline.run_submission(ds, cfg, num_boost_round=model.best_iteration)
    submission.build_submission(preds, ds.customers["customer_id"], ds.customer_ids).to_csv(SUB_PATH, index=False)

sub = pd.read_csv(SUB_PATH, dtype=str)
print(f"{len(sub):,} customers, {sub['prediction'].str.split().str.len().min()}-"
      f"{sub['prediction'].str.split().str.len().max()} predictions each")
sub.head()

1,371,980 customers, 12-12 predictions each


,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601006 0779781015 0568601044 0568601043 07...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0898713001 0898692006 0448509014 0781613006 06...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0794321007 0794321011 0918522001 0924243001 09...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0791587001 0852584001 0751471001 0866731001 09...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0791587001 0730683050 0791587015 0896152002 09...
